In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2025-01")
v_file_date = dbutils.widgets.get("p_file_date")
raw_race_path = f"{raw_folder_path}/{v_file_date}"
print(v_file_date, raw_race_path)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [0]:

results_schema_static = StructType([
    StructField("resultId", IntegerType(), False),
    StructField("raceId", IntegerType(), False),
    StructField("driverId", IntegerType(), False),
    StructField("constructorId", IntegerType(), False),
    StructField("number", IntegerType(), True),
    StructField("grid", IntegerType(), False),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), False),
    StructField("positionOrder", IntegerType(), False),
    StructField("points", DoubleType(), False),
    StructField("laps", IntegerType(), False),
    StructField("time", StringType(), True),
    StructField("milliseconds", IntegerType(), True),
    StructField("fastestLap", IntegerType(), True),
    StructField("rank", IntegerType(), True),
    StructField("fastestLapTime", StringType(), True),
    StructField("fastestLapSpeed", DoubleType(), True),
    StructField("statusId", StringType(), False),
])

results_schema_incremental = StructType([
    StructField("date", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True),
])

results_schema = results_schema_incremental if USE_INCREMENTAL else results_schema_static

if USE_INCREMENTAL:
    results_df = spark.read.schema(results_schema).json(f"{raw_race_path}/results/results_*.json")
else:
    results_df = spark.read.schema(results_schema).json(f"{raw_folder_path}/results.json")

display(results_df)

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, lit

if USE_INCREMENTAL:
    results_final_df = (
        results_df
        .withColumnRenamed("raceName", "race_name")
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("positionText", "position_text")
        .withColumn("ingestion_date", current_timestamp())
        .withColumn("data_source", lit(v_data_source))
        .withColumn("file_date", lit(v_file_date))
    )
    display(results_final_df.sort("season", "round", "position"))
else:
    results_final_df = (
        results_df
        .withColumnRenamed("resultId", "result_id")
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("positionText", "position_text")
        .withColumnRenamed("positionOrder", "position_order")
        .withColumnRenamed("fastestLap", "fastest_lap")
        .withColumnRenamed("fastestLapTime", "fastest_lap_time")
        .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed")
        .withColumn("ingestion_date", current_timestamp())
        .drop("statusId")
    )
    display(results_final_df.sort("race_id"))

In [0]:
from pyspark.sql.functions import col
if USE_INCREMENTAL:
    try:
        existing = spark.read.parquet(f"{processed_folder_path}/results")
        existing.filter(col("file_date") != v_file_date) \
            .write.mode("overwrite") \
            .partitionBy("season") \
            .format("delta") \
            .save(f"{processed_folder_path}/results")
    except Exception:
        pass

    results_final_df.write.mode("append") \
        .partitionBy("season") \
        .format("delta") \
        .save(f"{processed_folder_path}/results")

In [0]:
df = spark.read.format("delta").load(f"{processed_folder_path}/results")

display(df.groupBy("file_date").count().orderBy("file_date"))